# 2. Baseline modeling — 3D U-Net + transformer

Thin driver around the vendored baseline in `scripts/` (see
`docs/0_coding_standards.md` for why that logic lives in `scripts/` rather
than `src/` for now). Two independent things this notebook can do,
controlled by `RUN_MODE`:

- **`"submission"`**: predict on the real competition `test/` set and write
  `submission.csv`, for upload. Defaults to the baseline author's public
  pretrained checkpoint (`thibautgoldsborough/cellmot-baseline-artifacts`)
  so a first submission doesn't require training anything ourselves —
  see `docs/1_instructions.md`.
- **`"train"`**: train our own checkpoint from scratch (a documented next
  experiment, not required for a first submission).

Not yet run: needs the competition data, which isn't downloaded locally —
run on Kaggle via `scripts/push_kaggle_kernel.sh baseline` (competition
mount + the public `cellmot-baseline-artifacts` dataset, which bundles a
working `repo/` alongside pretrained `weights/`, auto-detect — see the
Setup cell) or point `$CELLMOT_DATA_DIR` at a local copy.

## 1. Setup & Config

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

IS_KAGGLE = Path("/kaggle").exists()


def _find_mount(candidates: list[Path], marker: str) -> Path | None:
    """Return the first candidate containing ``marker``, else scan /kaggle/input."""
    for c in candidates:
        if (c / marker).exists():
            return c
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for p in kaggle_input.glob(f"**/{marker}"):
            return p.parent
    return None


if IS_KAGGLE:
    # Pin numpy/scipy/torch to whatever Kaggle's base image already has:
    # letting pip pick a newer numpy for zarr>=3.0.10 breaks the base
    # image's precompiled scipy (ImportError deep in scipy.spatial/
    # numpy._core -- an ABI mismatch between new numpy and the untouched
    # old scipy build). Separately, tracksdata depends on torch, and an
    # unpinned reinstall swaps out the base image's GPU-driver-matched
    # torch build for an incompatible one (`CUDA error: no kernel image is
    # available for execution on the device`). Both observed directly on
    # this competition's Kaggle image. Must also run before importing
    # numpy/matplotlib/torch below, for the same reason.
    import importlib.metadata

    _pinned = {
        pkg: importlib.metadata.version(pkg) for pkg in ("numpy", "scipy", "torch")
    }
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            *(f"{pkg}=={ver}" for pkg, ver in _pinned.items()),
            "zarr>=3.0.10", "tqdm", "polars", "pyscipopt",
            "tracksdata @ git+https://github.com/royerlab/tracksdata@main",
        ],
        check=True,
    )

    ARTIFACTS_MOUNT = _find_mount(
        [
            Path("/kaggle/input/cellmot-baseline-artifacts"),
            Path("/kaggle/input/datasets/thibautgoldsborough/cellmot-baseline-artifacts/cellmot-baseline-artifacts"),
        ],
        "weights",
    )
    if ARTIFACTS_MOUNT is None:
        raise FileNotFoundError(
            "cellmot-baseline-artifacts dataset not found under /kaggle/input -- add "
            "it as a data source (see kernel-metadata.json)."
        )

    # The dataset mounts read-only, but the vendored scripts write
    # predictions/weights relative to their own file location
    # (scripts/dataspec.py) -- copy the repo to a writable location first.
    REPO_ROOT = Path("/kaggle/working/repo")
    if REPO_ROOT.exists():
        shutil.rmtree(REPO_ROOT)
    shutil.copytree(ARTIFACTS_MOUNT / "repo", REPO_ROOT)
else:
    REPO_ROOT = Path.cwd().parent
    ARTIFACTS_MOUNT = None  # pretrained weights are Kaggle-only; train locally instead

sys.path.insert(0, str(REPO_ROOT / "src"))
sys.path.insert(0, str(REPO_ROOT / "scripts"))
from dataspec import DATASET_PATH  # noqa: E402 -- needs REPO_ROOT on sys.path first

COMPETITION = "biohub-cell-tracking-during-development"
COMP_DIR = Path(f"/kaggle/input/competitions/{COMPETITION}")
TEST_DIR = COMP_DIR / "test" if IS_KAGGLE else REPO_ROOT / "data" / "test"

SEED = 0
RUN_MODE = "submission"  # "train" | "submission"

# --- "submission" mode: which weights to predict with ---------------------
USE_PRETRAINED = True  # True -> the public baseline checkpoint; False -> our own weights/ below
PRETRAINED_METHOD = "unet_transformer"
PRETRAINED_SPLIT = "0"

# --- "train" mode, and our-own-weights naming for "submission" mode -------
METHOD = "baseline"
SPLIT = "0"
EPOCHS = 3

# --- test-time detection/linking knobs (only used in "submission" mode) ---
# GT is sparse so the detector is poorly calibrated; ~0.99 scored best in the
# baseline author's sweep. ILP (global, flow-consistent linking) scored
# ~0.73 -> ~0.79 over the faster greedy linker in their notes.
DET_THRESHOLD = 0.99
UNET_BATCH_SIZE = 4
USE_ILP = True
ILP_EDGE_WEIGHT = -1.0
ILP_APPEARANCE_WEIGHT = 0.1
ILP_DISAPPEARANCE_WEIGHT = 0.1
ILP_DIVISION_WEIGHT = 1.0

# --- graph repair (post-ILP), see docs/3_strategy.md ----------------------
# Physical voxel scale (Z, Y, X), microns/voxel -- confirmed in docs/2_eda_insights.md.
SCALE_ZYX = (1.625, 0.40625, 0.40625)
# Both OFF by default: a real VALIDATE_ON_TRAIN_FOLD run (19 val videos,
# 2026-07-21) measured this config as *worse* than raw ILP output
# (edge_jaccard 0.8031 -> 0.7897, delta -0.0133) -- see docs/3_strategy.md.
# Code kept and still unit-tested; re-enable only after retuning against
# that regression (looser PRUNE_MIN_NODES, tighter GAP_MAX_DIST_UM, or
# isolating which of the two techniques is the actual culprit).
PRUNE_SHORT_TRACKS = False
PRUNE_MIN_NODES = 3        # drop connected components (tracks) with fewer nodes than this
CLOSE_GAPS = False
GAP_MAX = 2                # bridge dangling tracks missing up to this many consecutive frames
GAP_MAX_DIST_UM = 8.0      # max physical distance for a gap-closing match


def run(*args: str) -> None:
    """Run a vendored script with the current kernel's interpreter.

    Sets PYTHONPATH=<REPO_ROOT>/src so `import tracking_cellmot` resolves in
    the subprocess -- unlike a local `uv run`, nothing here `pip install -e`s
    the package, so it's only importable via sys.path/PYTHONPATH.
    """
    env = os.environ.copy()
    env["PYTHONPATH"] = str(REPO_ROOT / "src") + os.pathsep + env.get("PYTHONPATH", "")
    subprocess.run([sys.executable, *args], check=True, cwd=REPO_ROOT, env=env)


def resolve_weights() -> tuple[Path, str]:
    """Return (checkpoint path, method name) per USE_PRETRAINED."""
    if USE_PRETRAINED:
        if ARTIFACTS_MOUNT is None:
            raise FileNotFoundError(
                "USE_PRETRAINED=True but the cellmot-baseline-artifacts dataset isn't "
                "mounted -- add it as a data source, or set USE_PRETRAINED=False to use "
                "our own weights/ (requires RUN_MODE='train' first)."
            )
        split_dir = ARTIFACTS_MOUNT / "weights" / PRETRAINED_METHOD / f"split_{PRETRAINED_SPLIT}"
        return split_dir / "edge_predictor_best.pth", PRETRAINED_METHOD
    split_dir = REPO_ROOT / "weights" / METHOD / f"split_{SPLIT}"
    return split_dir / "edge_predictor_best.pth", METHOD


print(f"IS_KAGGLE={IS_KAGGLE}  REPO_ROOT={REPO_ROOT}")
if IS_KAGGLE:
    print(f"ARTIFACTS_MOUNT={ARTIFACTS_MOUNT}")

## 2. Graph repair (post-ILP)

Per `docs/3_strategy.md`: every top-scoring public approach reviewed —
learned or classical — adds a deterministic repair stage after
detection/linking. Implemented here, both **off by default** — a real
`VALIDATE_ON_TRAIN_FOLD` run (19 val videos, 2026-07-21) measured this
config as *worse* than raw ILP output (edge_jaccard 0.8031 → 0.7897,
delta -0.0133): our detector already runs at a strict `DET_THRESHOLD`, so
short predicted segments are less likely to be pure noise than in the
classical (DoG-detection) pipelines these techniques come from, and gap
closing may be introducing wrong bridges more often than recovering real
ones. Not shipped — see the finding at the end of this section and
`docs/3_strategy.md` for the retuning plan.

- **Short-track pruning** (`PRUNE_SHORT_TRACKS`): drop connected components
  (tracks, including division lineages) with fewer than `PRUNE_MIN_NODES`
  nodes.
- **Bounded gap closing** (`CLOSE_GAPS`, up to `GAP_MAX` frames): a track
  that ends early (no outgoing edge, not at the last timepoint) is bridged
  to a track that starts late (no incoming edge, not at the first
  timepoint) `gap + 1` frames later, if within `GAP_MAX_DIST_UM` — a
  physical-space Hungarian assignment per timepoint, one gap size at a
  time (1-frame gaps closed before 2-frame, so a loose 2-frame setting
  can't introduce edges a correct 1-frame match would have caught first).

Deferred for now (see `docs/3_strategy.md`'s roadmap): motion-aware
relinking (ILP already does global, flow-consistent linking, which is a
stronger starting point than the two-pass Hungarian these techniques
replace in detection-only public pipelines) and trajectory smoothing.

Unit-tested locally against synthetic data (7 cases: component grouping,
pruning, gap-distance gating, gap-size specificity, two-tier
gap1-then-gap2 sequencing, Hungarian conflict resolution) and against a
real `tracksdata` graph object before ever touching Kaggle — the code
runs correctly, it just doesn't help yet. Re-run `VALIDATE_ON_TRAIN_FOLD`
(section 6) after any retuning before re-enabling.

In [ ]:
import numpy as np
import polars as pl
import tracksdata as td
from scipy.optimize import linear_sum_assignment


def _connected_components(node_ids: list[int], edges: list[tuple[int, int]]) -> dict[int, int]:
    """Union-find over an edge list; returns node_id -> component root id."""
    parent = {n: n for n in node_ids}

    def find(x: int) -> int:
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    for s, t in edges:
        rs, rt = find(s), find(t)
        if rs != rt:
            parent[rs] = rt

    return {n: find(n) for n in node_ids}


def prune_short_tracks(nodes: pl.DataFrame, edges: pl.DataFrame, min_nodes: int) -> tuple[pl.DataFrame, pl.DataFrame]:
    """Drop nodes/edges belonging to connected components with fewer than min_nodes nodes."""
    if min_nodes <= 1 or nodes.height == 0:
        return nodes, edges
    comp = _connected_components(
        nodes["node_id"].to_list(),
        list(zip(edges["source_id"].to_list(), edges["target_id"].to_list(), strict=True)),
    )
    comp_series = pl.Series("_comp", [comp[n] for n in nodes["node_id"]])
    sizes = comp_series.value_counts()
    keep_comps = set(sizes.filter(pl.col("count") >= min_nodes)["_comp"].to_list())
    keep_nodes = {n for n, c in comp.items() if c in keep_comps}
    kept_nodes = nodes.filter(pl.col("node_id").is_in(list(keep_nodes)))
    kept_edges = edges.filter(
        pl.col("source_id").is_in(list(keep_nodes)) & pl.col("target_id").is_in(list(keep_nodes))
    )
    return kept_nodes, kept_edges


def close_gaps(
    nodes: pl.DataFrame,
    edges: pl.DataFrame,
    scale_zyx: tuple[float, float, float],
    gap: int,
    max_dist_um: float,
) -> pl.DataFrame:
    """Bridge `gap`-frame-missing tracks: dangling ends at t linked to dangling starts at t+gap+1."""
    if nodes.height == 0:
        return edges
    out_ids = set(edges["source_id"].to_list())
    in_ids = set(edges["target_id"].to_list())
    t_min, t_max = int(nodes["t"].min()), int(nodes["t"].max())

    ends = nodes.filter(~pl.col("node_id").is_in(list(out_ids)) & (pl.col("t") < t_max))
    starts = nodes.filter(~pl.col("node_id").is_in(list(in_ids)) & (pl.col("t") > t_min))

    scale = np.array(scale_zyx)
    new_edges = []
    used_starts: set[int] = set()
    for t in sorted(ends["t"].unique().to_list()):
        e_t = ends.filter(pl.col("t") == t)
        s_t = starts.filter((pl.col("t") == t + gap + 1) & (~pl.col("node_id").is_in(list(used_starts))))
        if e_t.height == 0 or s_t.height == 0:
            continue
        e_xyz = e_t.select(["z", "y", "x"]).to_numpy() * scale
        s_xyz = s_t.select(["z", "y", "x"]).to_numpy() * scale
        cost = np.linalg.norm(e_xyz[:, None, :] - s_xyz[None, :, :], axis=2)
        ri, ci = linear_sum_assignment(cost)
        e_ids = e_t["node_id"].to_list()
        s_ids = s_t["node_id"].to_list()
        for r, c in zip(ri, ci, strict=True):
            if cost[r, c] > max_dist_um:
                continue
            s_id = int(e_ids[int(r)])
            t_id = int(s_ids[int(c)])
            new_edges.append({"source_id": s_id, "target_id": t_id})
            used_starts.add(t_id)

    if not new_edges:
        return edges
    return pl.concat([edges, pl.DataFrame(new_edges, schema=edges.schema)])


def repair_graph(graph: "td.graph.BaseGraph") -> "td.graph.BaseGraph":
    """Apply gap-closing then short-track pruning to a predicted graph; returns a fresh graph.

    Reads the PRUNE_*/CLOSE_GAPS/GAP_*/SCALE_ZYX config above. Returns a new
    tracksdata InMemoryGraph -- the input graph is not mutated.
    """
    nodes = graph.node_attrs(attr_keys=["node_id", "t", "z", "y", "x"])
    edges = graph.edge_attrs(attr_keys=["source_id", "target_id"])

    if CLOSE_GAPS:
        for g in range(1, GAP_MAX + 1):
            edges = close_gaps(nodes, edges, SCALE_ZYX, gap=g, max_dist_um=GAP_MAX_DIST_UM)

    if PRUNE_SHORT_TRACKS:
        nodes, edges = prune_short_tracks(nodes, edges, PRUNE_MIN_NODES)

    out = td.graph.InMemoryGraph()
    for key in ("z", "y", "x"):
        out.add_node_attr_key(key, pl.Float64, 0.0)
    id_map: dict[int, int] = {}
    for row in nodes.iter_rows(named=True):
        new_id = out.add_node({"t": int(row["t"]), "z": float(row["z"]), "y": float(row["y"]), "x": float(row["x"])})
        id_map[row["node_id"]] = new_id
    for row in edges.iter_rows(named=True):
        s, t = id_map.get(row["source_id"]), id_map.get(row["target_id"])
        if s is not None and t is not None:
            out.add_edge(s, t, {})
    return out

**Finding (2026-07-21, 19 val videos):** repair made things worse.

| | edge_jaccard | division_jaccard | score |
|---|---:|---:|---:|
| raw (no repair) | 0.8031 | 0.0000 | 0.8031 |
| repaired | 0.7897 | 0.0000 | 0.7897 |
| delta | -0.0133 | +0.0000 | -0.0133 |

Both techniques are `False` by default above as a result. Division Jaccard
was 0 either way — this checkpoint isn't getting any division credit yet,
independent of repair. Next: isolate which technique (pruning vs. gap
closing) is the actual regression by toggling one at a time, and retry
with a looser `PRUNE_MIN_NODES` (2?) or a tighter `GAP_MAX_DIST_UM` (4-5µm?)
before assuming the whole approach is wrong — see `docs/3_strategy.md`.

## 3. Train (optional — skip if `USE_PRETRAINED`)

Only runs in `RUN_MODE == "train"`. Not needed for a first submission (see
`USE_PRETRAINED` above) — this is how to train our own checkpoint to try to
beat the public baseline later.

In [ ]:
if RUN_MODE == "train":
    run(
        "scripts/train_unet_transformer.py",
        "--split", SPLIT,
        "--epochs", str(EPOCHS),
    )
    print(f"Trained weights: {REPO_ROOT}/weights/{METHOD}/split_{SPLIT}/edge_predictor_best.pth")

*Insight: fill in after running — training loss curve, whether it converged
in `EPOCHS` epochs, any stability issues.*

## 4. Predict on the competition test set

Only runs in `RUN_MODE == "submission"`. `predict_unet_transformer.py`
requires a `dataset_splits.json` listing which videos to predict — the real
`test/` directory doesn't ship one (that's a train-only, fold-splitting
concept), so build a synthetic one-fold file listing every test video
first, matching the approach in the baseline author's own public inference
notebook (`thibautgoldsborough/unet-baseline-inference-submission`).

In [ ]:
if RUN_MODE == "submission":
    import json

    test_stems = sorted(p.stem for p in TEST_DIR.glob("*.zarr"))
    print(f"{len(test_stems)} test videos under {TEST_DIR}")

    test_splits_file = REPO_ROOT / "kaggle_test_splits.json"
    test_splits_file.write_text(json.dumps([{"split": 0, "train": [], "test": test_stems}]))

    weights_path, predict_method = resolve_weights()

    predict_args = [
        "scripts/predict_unet_transformer.py",
        "--data-dir", str(TEST_DIR),
        "--splits", str(test_splits_file),
        "--split", "0",
        "--method", predict_method,
        "--weights", str(weights_path),
        "--unet-batch-size", str(UNET_BATCH_SIZE),
        "--det-threshold", str(DET_THRESHOLD),
        "--ilp-edge-weight", str(ILP_EDGE_WEIGHT),
        "--ilp-appearance-weight", str(ILP_APPEARANCE_WEIGHT),
        "--ilp-disappearance-weight", str(ILP_DISAPPEARANCE_WEIGHT),
        "--ilp-division-weight", str(ILP_DIVISION_WEIGHT),
    ]
    if USE_ILP:
        predict_args.append("--use-ilp")

    run(*predict_args)

## 5. Build `submission.csv`

Applies graph repair (section 2) to each predicted `.geff`, then flattens
the repaired graphs into the competition's CSV schema — verified against
the real `sample_submission.csv` downloaded via the Kaggle CLI:
`id,dataset,row_type,node_id,t,z,y,x,source_id,target_id`, one `node` row
per detection and one `edge` row per link.

Flattening is inlined rather than calling `scripts/geffs_to_csv.py`: the
artifacts dataset's bundled `repo/` only includes what the baseline
author's own inference notebook needs (train/predict/dataspec), not this
project's extra conversion scripts (`geffs_to_csv.py`, `csv_to_geffs.py`,
`evaluate.py`) — confirmed missing on a real run. This mirrors the exact
logic in `scripts/geffs_to_csv.py`, kept in sync by hand for now.

In [ ]:
if RUN_MODE == "submission":
    kaggle_user = os.environ.get("USER", os.environ.get("USERNAME", "unknown"))
    predictions_dir = REPO_ROOT / "predictions" / kaggle_user / predict_method / "split_0"
    submission_csv = Path("/kaggle/working/submission.csv") if IS_KAGGLE else REPO_ROOT / "submission.csv"

    def _graph_to_rows(graph, name: str) -> pl.DataFrame:
        """Flatten one graph into node rows then edge rows (submission schema)."""
        nodes = graph.node_attrs().select(
            pl.lit(name).alias("dataset"),
            pl.lit("node").alias("row_type"),
            pl.col("node_id").cast(pl.Int64),
            pl.col("t").cast(pl.Int64),
            pl.col("z").cast(pl.Float64).round(0).cast(pl.Int64),
            pl.col("y").cast(pl.Float64).round(0).cast(pl.Int64),
            pl.col("x").cast(pl.Float64).round(0).cast(pl.Int64),
            pl.lit(-1, dtype=pl.Int64).alias("source_id"),
            pl.lit(-1, dtype=pl.Int64).alias("target_id"),
        )
        edges = graph.edge_attrs().select(
            pl.lit(name).alias("dataset"),
            pl.lit("edge").alias("row_type"),
            pl.lit(-1, dtype=pl.Int64).alias("node_id"),
            pl.lit(-1, dtype=pl.Int64).alias("t"),
            pl.lit(-1, dtype=pl.Int64).alias("z"),
            pl.lit(-1, dtype=pl.Int64).alias("y"),
            pl.lit(-1, dtype=pl.Int64).alias("x"),
            pl.col("source_id").cast(pl.Int64),
            pl.col("target_id").cast(pl.Int64),
        )
        return pl.concat([nodes, edges])

    geffs = sorted(predictions_dir.glob("*.geff"))
    frames = []
    for g in geffs:
        graph = td.graph.IndexedRXGraph.from_geff(str(g))
        graph = graph[0] if isinstance(graph, tuple) else graph
        n_before, e_before = graph.num_nodes(), graph.num_edges()
        graph = repair_graph(graph)
        frames.append(_graph_to_rows(graph, g.stem))
        print(
            f"{g.stem}: {n_before} nodes, {e_before} edges "
            f"-> repaired: {graph.num_nodes()} nodes, {graph.num_edges()} edges"
        )

    columns = ["dataset", "row_type", "node_id", "t", "z", "y", "x", "source_id", "target_id"]
    table = pl.concat(frames) if frames else pl.DataFrame(schema=dict.fromkeys(columns, pl.Int64))
    table = table.with_row_index("id")
    table.write_csv(submission_csv)
    print(f"Wrote {table.height} rows from {len(geffs)} geffs to {submission_csv}")

## 6. (Optional) Validate methodology on a train fold

The real `test/` set has no local ground truth to score against. To
sanity-check the weights/detection/linking/**repair** config *before*
spending a submission attempt, predict on a held-out **train** fold instead
(real GT available) and score locally with the competition's own metric
(`docs/1_instructions.md` / `metrics.md`) — both **with and without graph
repair**, so the repair stage's actual effect is visible before it's
trusted. Builds its own deterministic 90/10 train/val split the same way
`train_unet_transformer.py` does when no `dataset_splits.json` is present,
so it doesn't depend on a prior run.

In [ ]:
VALIDATE_ON_TRAIN_FOLD = False  # set True to sanity-check before submitting

if VALIDATE_ON_TRAIN_FOLD:
    import json
    import random

    from tracking_cellmot.io import open_dataset
    from tracking_cellmot.metrics import evaluate_datasets

    stems = sorted(
        p.name[:-5] for p in DATASET_PATH.glob("*.zarr")
        if (DATASET_PATH / f"{p.name[:-5]}.geff").exists()
    )
    random.Random(0).shuffle(stems)
    n_val = max(1, len(stems) // 10)
    val_stems = stems[:n_val]
    train_splits_file = REPO_ROOT / "kaggle_train_splits.json"
    train_splits_file.write_text(json.dumps(
        [{"split": 0, "train": stems[n_val:], "test": val_stems}]
    ))
    print(f"{len(stems) - n_val} train / {n_val} val videos under {DATASET_PATH}")

    weights_path, validate_method = resolve_weights()

    predict_args = [
        "scripts/predict_unet_transformer.py",
        "--data-dir", str(DATASET_PATH),
        "--splits", str(train_splits_file),
        "--split", "0",
        "--method", validate_method,
        "--weights", str(weights_path),
        "--unet-batch-size", str(UNET_BATCH_SIZE),
        "--det-threshold", str(DET_THRESHOLD),
        "--ilp-edge-weight", str(ILP_EDGE_WEIGHT),
        "--ilp-appearance-weight", str(ILP_APPEARANCE_WEIGHT),
        "--ilp-disappearance-weight", str(ILP_DISAPPEARANCE_WEIGHT),
        "--ilp-division-weight", str(ILP_DIVISION_WEIGHT),
    ]
    if USE_ILP:
        predict_args.append("--use-ilp")

    run(*predict_args)  # writes one .geff per val video -- no --evaluate, we score below ourselves

    kaggle_user = os.environ.get("USER", os.environ.get("USERNAME", "unknown"))
    val_predictions_dir = REPO_ROOT / "predictions" / kaggle_user / validate_method / "split_0"

    raw_pairs = []
    repaired_pairs = []
    for stem in val_stems:
        geff_path = val_predictions_dir / f"{stem}.geff"
        if not geff_path.exists():
            print(f"  {stem}: no prediction found, skipped")
            continue
        pred_graph = td.graph.IndexedRXGraph.from_geff(str(geff_path))
        pred_graph = pred_graph[0] if isinstance(pred_graph, tuple) else pred_graph
        gt_graph = open_dataset(
            DATASET_PATH / stem, normalize=False, load_image=False, require_tracks=True
        ).tracks
        raw_pairs.append((pred_graph, gt_graph))
        repaired_pairs.append((repair_graph(pred_graph), gt_graph))

    raw_result = evaluate_datasets(raw_pairs, scale=SCALE_ZYX)
    repaired_result = evaluate_datasets(repaired_pairs, scale=SCALE_ZYX)

    header = f"{'':20s} {'edge_jaccard':>14s} {'division_jaccard':>18s} {'score':>10s}"
    print(f"\n{header}")
    print(
        f"{'raw (no repair)':20s} {raw_result.edge_jaccard:14.4f} "
        f"{raw_result.division_jaccard:18.4f} {raw_result.score:10.4f}"
    )
    print(
        f"{'repaired':20s} {repaired_result.edge_jaccard:14.4f} "
        f"{repaired_result.division_jaccard:18.4f} {repaired_result.score:10.4f}"
    )
    print(
        f"{'delta':20s} {repaired_result.edge_jaccard - raw_result.edge_jaccard:+14.4f} "
        f"{repaired_result.division_jaccard - raw_result.division_jaccard:+18.4f} "
        f"{repaired_result.score - raw_result.score:+10.4f}"
    )

*Insight: already run 2026-07-21 on 19 val videos — see section 2's
finding table (repair currently hurts, -0.0133). Re-run this cell after
retuning `PRUNE_MIN_NODES`/`GAP_MAX_DIST_UM` or isolating which technique
regressed, before re-enabling either flag or spending a submission.*

## Submitting to Kaggle

This competition scores a **file upload**, not a notebook rerun (see
`docs/1_instructions.md`) — after `RUN_MODE == "submission"` finishes on
Kaggle, download `submission.csv` from the kernel's Output tab, then
either upload it on the competition's Submit page, or from a shell with
Kaggle CLI access:

```bash
uv run kaggle competitions submit \
    -c biohub-cell-tracking-during-development \
    -f submission.csv \
    -m "unet_transformer split_0 pretrained, ILP, det-threshold 0.99"
```

Not run automatically from this notebook — submissions count against a
daily quota and should be a deliberate action, not a side effect of
re-running a cell.

## Findings / limitations / next experiment

- **Findings** (real runs, 2026-07-21): predict + submission pipeline
  completes end-to-end on Kaggle GPU (`device=cuda`, `NvidiaTeslaT4`) using
  the pretrained `unet_transformer` checkpoint. Graph repair (section 2)
  was validated on 19 real val videos and **currently hurts** — edge
  Jaccard 0.8031 → 0.7897 (Δ -0.0133), division Jaccard 0 either way — so
  it's off by default; only the plain ILP output is currently recommended
  for submission. This is exactly why `VALIDATE_ON_TRAIN_FOLD` exists:
  catching a regression before it costs a real submission attempt.
- **Limitations**: the pretrained checkpoint (`unet_transformer`, split 0)
  wasn't trained to convergence per the baseline author's own notes.
  `DET_THRESHOLD`/ILP weights above are their reported best settings, not
  necessarily ours. Graph repair's regression isn't root-caused yet —
  section 2's finding notes the leading hypothesis (our detector is
  stricter than the classical pipelines these techniques come from, so
  "short tracks" are less likely to be pure noise here) and next steps
  (isolate pruning vs. gap-closing, retune thresholds).
- **Next** — see `docs/3_strategy.md` for the full prioritized roadmap.
  1. Get a real submission banked with repair *off* (current best-known
     config) rather than blocking on repair working.
  2. Root-cause the repair regression: toggle `PRUNE_SHORT_TRACKS` and
     `CLOSE_GAPS` independently via `VALIDATE_ON_TRAIN_FOLD` to find which
     one (or both) regresses, then retune (`PRUNE_MIN_NODES`,
     `GAP_MAX_DIST_UM`) rather than discarding the approach outright.
  3. Read `estimated_number_of_nodes` (now in `01_eda.ipynb`) to calibrate
     `DET_THRESHOLD` against an over-prediction budget.
  4. Division recovery is low-value in isolation (10% metric weight,
     `metrics.md`) — only after 1–3 are solid.
  5. Training our own checkpoint (`RUN_MODE = "train"`, `USE_PRETRAINED =
     False`), motion-aware relinking, trajectory smoothing, or D4
     test-time augmentation are later-stage refinements.